# Fish Speech Pipeline Worker — Colab GPU

Runs the **Fish Speech worker** on Colab's free GPU (T4).  
Fish Speech 1.5 produces high-quality, natural English narration via zero-shot voice cloning.

## Before you start

**On your local machine**, expose Redis so Colab can reach it:
```bash
# Option A: ngrok (recommended)
ngrok tcp 6379
# → copy the  tcp://X.tcp.ngrok.io:XXXXX  URL

# Option B: cloudflared
cloudflared tunnel --url tcp://localhost:6379
```

**Set Colab Secrets** (🔑 icon, left sidebar):

| Secret | Value |
|--------|-------|
| `NGROK_TOKEN` | ngrok authtoken (dashboard.ngrok.com → Your Authtoken) |
| `SSH_HOST` | server IP / hostname |
| `SSH_USER` | SSH username |
| `SSH_KEY`  | full private key (paste `-----BEGIN ... KEY-----`) |
| `SSH_REMOTE_PATH` | remote output dir, e.g. `/opt/tts-node/outputs` |
| `SSH_PORT` | SSH port (default 22) |
| `GITHUB_TOKEN` | PAT for private repos (optional) |
| `GITHUB_USER`  | GitHub username (optional) |

**Reference audio** (for voice cloning) is loaded from Google Drive.  
Upload a clean 10–30 s narrator WAV to `My Drive/tts-pipeline/reference.wav`  
and set `FISH_REF_TEXT` in cell 4 to the exact words spoken in that file.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ── 1. Mount Google Drive (MP3 output + reference audio storage) ──────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/tts-pipeline/outputs'
SPOOL_DIR  = '/content/spool'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPOOL_DIR,  exist_ok=True)
os.makedirs('/content/ref', exist_ok=True)
print('Drive mounted. Output dir:', OUTPUT_DIR)

In [ ]:
# ── 2. Secrets ─────────────────────────────────────────────────────────────────
# Add these in the Secrets panel (🔑 icon, left sidebar):
#
#   GITHUB_TOKEN      → ghp_...  (Personal Access Token — private repos only)
#   GITHUB_USER       → your GitHub username
#   NGROK_TOKEN       → ngrok authtoken (dashboard.ngrok.com → Your Authtoken)
#
#   SSH_HOST          → server IP or hostname  e.g. 10.0.0.1
#   SSH_USER          → SSH username           e.g. root
#   SSH_KEY           → private key contents   (paste the full -----BEGIN ... KEY-----)
#   SSH_REMOTE_PATH   → remote output dir      e.g. /opt/tts-node/outputs
#   SSH_PORT          → SSH port (optional, default 22)
#
from google.colab import userdata

GITHUB_TOKEN    = userdata.get('GITHUB_TOKEN')
GITHUB_USER     = userdata.get('GITHUB_USER')
NGROK_TOKEN     = userdata.get('NGROK_TOKEN')
SSH_HOST        = userdata.get('SSH_HOST')
SSH_USER        = userdata.get('SSH_USER')
SSH_KEY         = userdata.get('SSH_KEY')
SSH_REMOTE_PATH = userdata.get('SSH_REMOTE_PATH') or '/opt/tts-node/outputs'
SSH_PORT        = int(userdata.get('SSH_PORT') or 22)

if GITHUB_TOKEN and GITHUB_USER:
    import subprocess
    subprocess.run([
        'git', 'config', '--global', 'credential.helper',
        f'!f() {{ echo username={GITHUB_USER}; echo password={GITHUB_TOKEN}; }}; f'
    ], check=True)
    print(f'Git auth: {GITHUB_USER}')
else:
    print('Git auth: skipped (public repo)')

print(f'ngrok token:  {"ok" if NGROK_TOKEN else "MISSING"}')
print(f'SSH host:     {"ok" if SSH_HOST    else "MISSING"}')
print(f'SSH key:      {"ok" if SSH_KEY     else "MISSING"}')

In [ ]:
# ── 3. SSH setup + mount remote output dir via sshfs ──────────────────────────
import os, subprocess, textwrap, re

if not (SSH_HOST and SSH_USER and SSH_KEY):
    raise RuntimeError('SSH_HOST, SSH_USER and SSH_KEY must all be set in Secrets.')

SSH_KEY_PATH = '/root/.ssh/colab_key'
os.makedirs('/root/.ssh', exist_ok=True)

def _reconstruct_pem(raw: str) -> str:
    key = raw.replace('\\n', '\n').strip()
    pattern = r'(-----BEGIN [^-]+-----)(.*?)(-----END [^-]+-----)'
    match = re.search(pattern, key, re.DOTALL)
    if match:
        header, body, footer = match.groups()
        body = ''.join(body.split())
        wrapped = '\n'.join(body[i:i+64] for i in range(0, len(body), 64))
        return f'{header}\n{wrapped}\n{footer}\n'
    return key + '\n'

with open(SSH_KEY_PATH, 'w') as f:
    f.write(_reconstruct_pem(SSH_KEY))
os.chmod(SSH_KEY_PATH, 0o600)

check = subprocess.run(['ssh-keygen', '-y', '-f', SSH_KEY_PATH], capture_output=True, text=True)
if check.returncode != 0:
    print('Debug - First 100 chars of processed key:\n', open(SSH_KEY_PATH).read()[:100])
    raise RuntimeError(f'SSH Key Error: {check.stderr}')

with open('/root/.ssh/config', 'w') as f:
    f.write(textwrap.dedent(f"""\
        Host {SSH_HOST}
            StrictHostKeyChecking no
            UserKnownHostsFile /dev/null
            IdentityFile {SSH_KEY_PATH}
            Port {SSH_PORT}
            ConnectTimeout 10
    """))

print(f'SSH key verified. Testing connection to {SSH_USER}@{SSH_HOST}...')
test_conn = subprocess.run(
    ['ssh', f'{SSH_USER}@{SSH_HOST}', f'mkdir -p {SSH_REMOTE_PATH}'],
    capture_output=True, text=True
)
if test_conn.returncode != 0:
    raise RuntimeError(f'Connection failed: {test_conn.stderr}')

!apt-get install -y sshfs > /dev/null 2>&1
SSHFS_MOUNT = '/content/remote-outputs'
os.makedirs(SSHFS_MOUNT, exist_ok=True)
subprocess.run(['fusermount', '-uz', SSHFS_MOUNT], capture_output=True)

mount_cmd = [
    'sshfs', f'{SSH_USER}@{SSH_HOST}:{SSH_REMOTE_PATH}', SSHFS_MOUNT,
    '-o', f'IdentityFile={SSH_KEY_PATH},port={SSH_PORT}',
    '-o', 'reconnect,ServerAliveInterval=15,ServerAliveCountMax=3'
]
result = subprocess.run(mount_cmd, capture_output=True, text=True)
if result.returncode == 0:
    OUTPUT_DIR = SSHFS_MOUNT
    os.environ['OUTPUT_DIR'] = OUTPUT_DIR
    print(f'Mounted {SSH_REMOTE_PATH} → {SSHFS_MOUNT}')
else:
    print(f'sshfs mount failed: {result.stderr}')
    print('Falling back to Google Drive output.')

In [ ]:
# ── 4. Config — edit these values ─────────────────────────────────────────────
REDIS_URL = 'rediss://redis.example.com:6380'  # ← your Redis URL (rediss:// for TLS)
WORKER_ID = 'colab-fish-1'

# Fish Speech model to download from HuggingFace.
# fish-speech-1.5 fits on T4 (15 GB VRAM) and produces high-quality English.
FISH_MODEL_REPO = 'fishaudio/fish-speech-1.5'
FISH_MODEL_DIR  = '/content/fish-speech/checkpoints/fish-speech-1.5'

# Reference audio for zero-shot voice cloning.
# Upload a clean 10–30 s narrator WAV to Google Drive first (see cell 7).
# Set FISH_REF_TEXT to the exact words spoken in the WAV file.
FISH_REF_AUDIO = '/content/ref/reference.wav'
FISH_REF_TEXT  = 'He was an old man who fished alone in a skiff in the Gulf Stream.'

# Fish Speech API server port (runs locally inside the Colab VM).
FISH_API_PORT = 8080

# Number of inference steps (higher = slightly more natural, slower).
FISH_NUM_SAMPLES = 1

os.environ['REDIS_URL']         = REDIS_URL
os.environ['WORKER_ID']         = WORKER_ID
os.environ['OUTPUT_DIR']        = OUTPUT_DIR
os.environ['SPOOL_DIR']         = SPOOL_DIR
os.environ['FISH_MODEL_DIR']    = FISH_MODEL_DIR
os.environ['FISH_REF_AUDIO']    = FISH_REF_AUDIO
os.environ['FISH_REF_TEXT']     = FISH_REF_TEXT
os.environ['FISH_API_PORT']     = str(FISH_API_PORT)
os.environ['FISH_NUM_SAMPLES']  = str(FISH_NUM_SAMPLES)
os.environ['PROMETHEUS_PORT']   = '8005'
print('Config set.')

In [ ]:
# ── 5. Install system dependencies ────────────────────────────────────────────
!apt-get install -y ffmpeg build-essential git > /dev/null 2>&1
print('ffmpeg + build-essential + git installed.')

In [ ]:
# ── 6. Install Fish Speech + GPU PyTorch ──────────────────────────────────────
# Colab T4 runs CUDA 12.x — use the cu121 wheel for PyTorch.
# Fish Speech 1.5 uses a Llama-based architecture (~500 M active params on T4).
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121

# Clone the fish-speech repo so we get the API server and all tools.
import subprocess, os
result = subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/fishaudio/fish-speech',
    '/content/fish-speech'
], capture_output=True, text=True)
print(result.stdout or result.stderr)

# Install the package in editable mode so the tools/ scripts can import it.
!pip install -q -e /content/fish-speech
!pip install -q soundfile pydub redis prometheus_client huggingface_hub
print('All packages installed.')

In [ ]:
# ── 7. Download Fish Speech 1.5 model weights ─────────────────────────────────
# ~500 MB download; cached in /content after first run.
# For persistent caching across sessions, point HF_HOME at Google Drive:
#   os.environ['HF_HOME'] = '/content/drive/MyDrive/tts-pipeline/hf-cache'

import os
from huggingface_hub import snapshot_download

if not os.path.exists(os.path.join(FISH_MODEL_DIR, 'model.pth')) and \
   not os.path.exists(os.path.join(FISH_MODEL_DIR, 'consolidated.00.pth')):
    print(f'Downloading {FISH_MODEL_REPO} → {FISH_MODEL_DIR} ...')
    snapshot_download(
        repo_id=FISH_MODEL_REPO,
        local_dir=FISH_MODEL_DIR,
        ignore_patterns=['*.md', '*.txt', 'original/*'],
    )
    print('Download complete.')
else:
    print(f'Model weights already present at {FISH_MODEL_DIR}')

# List the checkpoint files so we can confirm what was downloaded.
import glob
ckpts = glob.glob(os.path.join(FISH_MODEL_DIR, '*.pth')) + \
        glob.glob(os.path.join(FISH_MODEL_DIR, '*.ckpt'))
for f in sorted(ckpts):
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  {os.path.basename(f):60s}  {size_mb:>8.1f} MB')

In [ ]:
# ── 8. Reference audio ────────────────────────────────────────────────────────
# Fish Speech clones the voice from a short reference WAV (10–30 s).
# For best results use clean, noise-free narration — no music or background.
#
# OPTION A — load from Google Drive (recommended, persists across sessions):
#   Upload your WAV to Drive → My Drive/tts-pipeline/reference.wav
#   Set FISH_REF_TEXT in cell 4 to the exact words spoken in that file.
#
# OPTION B — fall back to a bundled LJ Speech sample (US female narrator).

import os, shutil

DRIVE_REF_WAV = '/content/drive/MyDrive/tts-pipeline/reference.wav'

if os.path.exists(DRIVE_REF_WAV):
    shutil.copy(DRIVE_REF_WAV, FISH_REF_AUDIO)
    print(f'Loaded reference audio from Drive ({os.path.getsize(FISH_REF_AUDIO):,} bytes)')
    print(f'Ref text: "{FISH_REF_TEXT}"')

else:
    # Download a short LJ Speech sample as a usable fallback.
    print('Drive reference not found — downloading LJ Speech sample...')
    import urllib.request
    lj_url = (
        'https://data.keithito.com/data/speech/'
        'LJSpeech-1.1/wavs/LJ001-0001.wav'
    )
    try:
        urllib.request.urlretrieve(lj_url, FISH_REF_AUDIO)
        FISH_REF_TEXT = (
            'Printing, in the only sense with which we are at present concerned, '
            'differs from most if not from all the arts and crafts represented in '
            'the Exhibition'
        )
        os.environ['FISH_REF_TEXT'] = FISH_REF_TEXT
        print(f'LJ Speech sample saved ({os.path.getsize(FISH_REF_AUDIO):,} bytes)')
        print('Tip: for better results upload your own narrator WAV to Google Drive.')
    except Exception as e:
        print(f'Could not download LJ sample: {e}')
        print('Place a reference WAV at /content/ref/reference.wav manually.')

if not os.path.exists(FISH_REF_AUDIO):
    raise RuntimeError(f'No reference audio at {FISH_REF_AUDIO}. Worker cannot start.')

os.environ['FISH_REF_AUDIO'] = FISH_REF_AUDIO
print('Reference audio ready.')

In [ ]:
# ── 9. Start Fish Speech HTTP API server ──────────────────────────────────────
# The server runs in the background. The worker (cell 12) calls it via HTTP.
# Logs are streamed to /content/fish-api.log.

import subprocess, time, os, sys

LOG_PATH = '/content/fish-api.log'

# Build the launch command.
# --half:    fp16 inference — essential to fit within T4's 15 GB VRAM.
# --compile: disabled — torch.compile has a long warm-up that wastes free-tier time.
api_cmd = [
    sys.executable,
    '/content/fish-speech/tools/api_server.py',
    '--listen', f'0.0.0.0:{FISH_API_PORT}',
    '--llama-checkpoint-path', FISH_MODEL_DIR,
    '--decoder-checkpoint-path',
        os.path.join(FISH_MODEL_DIR, 'firefly-gan-vq-fsq-8x1024-21hz-generator.pth'),
    '--half',
]

print(f'Starting Fish Speech API server on port {FISH_API_PORT}...')
print(f'Logs: {LOG_PATH}')

with open(LOG_PATH, 'w') as log_f:
    _api_proc = subprocess.Popen(
        api_cmd,
        stdout=log_f,
        stderr=subprocess.STDOUT,
        cwd='/content/fish-speech',
    )

# Wait for the server to become healthy (model loading takes ~60 s on T4).
import urllib.request, urllib.error

HEALTH_URL   = f'http://localhost:{FISH_API_PORT}/'
TIMEOUT_S    = 180
POLL_INTERVAL = 5

print(f'Waiting for server (up to {TIMEOUT_S}s)...', end='', flush=True)
deadline = time.time() + TIMEOUT_S
while time.time() < deadline:
    try:
        urllib.request.urlopen(HEALTH_URL, timeout=3)
        print(' ready!')
        break
    except Exception:
        # Print last log line for progress visibility
        try:
            last = subprocess.check_output(['tail', '-1', LOG_PATH], text=True).strip()
            print(f'\r  {last[:80]:80s}', end='', flush=True)
        except Exception:
            pass
        time.sleep(POLL_INTERVAL)
else:
    print('\nServer did not start in time. Last 20 log lines:')
    !tail -20 /content/fish-api.log
    raise RuntimeError('Fish Speech API server failed to start.')

print(f'Fish Speech API server running (PID {_api_proc.pid}).')

In [ ]:
# ── 10. Smoke test the Fish Speech API ────────────────────────────────────────
import base64, os, time
import urllib.request, urllib.error, json

with open(FISH_REF_AUDIO, 'rb') as f:
    _ref_b64 = base64.b64encode(f.read()).decode()

_payload = json.dumps({
    'text': 'Pipeline worker ready.',
    'references': [{'audio': _ref_b64, 'text': FISH_REF_TEXT}],
    'format': 'mp3',
    'streaming': False,
    'normalize': True,
    'num_samples': int(FISH_NUM_SAMPLES),
}).encode()

_req = urllib.request.Request(
    f'http://localhost:{FISH_API_PORT}/v1/tts',
    data=_payload,
    headers={'Content-Type': 'application/json'},
    method='POST',
)

print('Running smoke-test synthesis...', end='', flush=True)
_t0 = time.time()
try:
    with urllib.request.urlopen(_req, timeout=120) as _resp:
        _audio = _resp.read()
    _elapsed = time.time() - _t0
    print(f' OK  ({len(_audio):,} bytes in {_elapsed:.1f}s)')
except urllib.error.HTTPError as e:
    body = e.read().decode(errors='replace')
    raise RuntimeError(f'Fish Speech API error {e.code}: {body}')

In [ ]:
# ── 11. Verify Redis + output directory ───────────────────────────────────────
import redis, os
from pathlib import Path

r = redis.from_url(os.environ['REDIS_URL'], decode_responses=True, socket_connect_timeout=5)
print('Redis ping:', r.ping())
print('TTS queue depth:', r.llen('pipeline:tts'))

out = Path(os.environ['OUTPUT_DIR'])
out.mkdir(parents=True, exist_ok=True)
probe = out / '.write_test'
try:
    probe.touch()
    probe.unlink()
    print(f'Output dir writable:  {out}  ✓')
except OSError as e:
    print(f'Output dir NOT writable: {out}  ✗  ({e})')
    print('  → chunks will spool to', os.environ['SPOOL_DIR'])

spool = Path(os.environ['SPOOL_DIR'])
spooled = list(spool.rglob('*.mp3'))
if spooled:
    print(f'WARNING: {len(spooled)} MP3(s) stuck in spool from a previous run:')
    for f in spooled[:5]:
        print(f'  {f}')
else:
    print(f'Spool clean: {spool}')

In [ ]:
# ── 12. Start the Fish Speech worker ──────────────────────────────────────────
# Pulls chunk jobs from pipeline:tts, calls the local Fish Speech API,
# writes MP3s to OUTPUT_DIR (or spools to SPOOL_DIR if the mount is offline).
# Runs until the session ends or you interrupt the cell.

import base64, json, logging, os, shutil, signal, sys, time, urllib.request, urllib.error
from pathlib import Path

import redis
from prometheus_client import Counter, Gauge, Histogram, start_http_server

# ── logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('fish-worker')

# ── config ────────────────────────────────────────────────────────────────────
REDIS_URL      = os.environ['REDIS_URL']
WORKER_ID      = os.environ.get('WORKER_ID', 'colab-fish-1')
OUTPUT_DIR_W   = Path(os.environ['OUTPUT_DIR'])
SPOOL_DIR_W    = Path(os.environ['SPOOL_DIR'])
FISH_API_BASE  = f'http://localhost:{os.environ["FISH_API_PORT"]}'
REF_AUDIO      = os.environ['FISH_REF_AUDIO']
REF_TEXT       = os.environ['FISH_REF_TEXT']
NUM_SAMPLES    = int(os.environ.get('FISH_NUM_SAMPLES', '1'))
TTS_TIMEOUT    = int(os.environ.get('TTS_TIMEOUT_S', '300'))
SMB_RETRY      = int(os.environ.get('SMB_RETRY_INTERVAL', '30'))
PROM_PORT      = int(os.environ.get('PROMETHEUS_PORT', '8005'))

OUTPUT_DIR_W.mkdir(parents=True, exist_ok=True)
SPOOL_DIR_W.mkdir(parents=True, exist_ok=True)

# ── Prometheus ────────────────────────────────────────────────────────────────
try:
    start_http_server(PROM_PORT)
    log.info('Prometheus metrics on :%d', PROM_PORT)
except Exception:
    pass  # port already in use from a previous run

_jobs_total    = Counter('fish_worker_jobs_total',    'Jobs processed', ['status', 'worker_id'])
_api_latency   = Histogram('fish_worker_api_latency_seconds', 'Fish API latency', ['worker_id'])
_heartbeat     = Gauge('fish_worker_heartbeat_timestamp', 'Last heartbeat', ['worker_id'])
_status_gauge  = Gauge('fish_worker_status', 'Worker state', ['worker_id', 'state'])

def _set_status(state: str):
    for s in ('Idle', 'Processing', 'Error'):
        _status_gauge.labels(worker_id=WORKER_ID, state=s).set(1 if s == state else 0)

_set_status('Idle')

# ── pre-encode reference audio for every API call ────────────────────────────
with open(REF_AUDIO, 'rb') as _f:
    _ref_audio_b64 = base64.b64encode(_f.read()).decode()
log.info('Reference audio ready (%d bytes)', os.path.getsize(REF_AUDIO))

# ── synthesis helper ─────────────────────────────────────────────────────────
def synthesize_text(text: str, speed: float = 1.0) -> bytes:
    """Call Fish Speech API; return raw MP3 bytes."""
    payload = json.dumps({
        'text': text,
        'references': [{'audio': _ref_audio_b64, 'text': REF_TEXT}],
        'format': 'mp3',
        'streaming': False,
        'normalize': True,
        'num_samples': NUM_SAMPLES,
    }).encode()

    req = urllib.request.Request(
        f'{FISH_API_BASE}/v1/tts',
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req, timeout=TTS_TIMEOUT) as resp:
        return resp.read()

# ── output path helper ────────────────────────────────────────────────────────
def _output_path(book_id: str, chunk_idx: int) -> Path:
    book_dir = OUTPUT_DIR_W / book_id
    try:
        book_dir.mkdir(parents=True, exist_ok=True)
        probe = book_dir / '.write_test'
        probe.touch()
        probe.unlink()
        return book_dir / f'{chunk_idx:05d}.mp3'
    except OSError:
        spool_dir = SPOOL_DIR_W / book_id
        spool_dir.mkdir(parents=True, exist_ok=True)
        return spool_dir / f'{chunk_idx:05d}.mp3'

def _flush_spool(book_id: str):
    spool_dir = SPOOL_DIR_W / book_id
    if not spool_dir.exists():
        return
    dest_dir = OUTPUT_DIR_W / book_id
    for f in sorted(spool_dir.glob('*.mp3')):
        try:
            dest_dir.mkdir(parents=True, exist_ok=True)
            shutil.move(str(f), dest_dir / f.name)
            log.info('Flushed spooled chunk %s → %s', f.name, dest_dir)
        except OSError:
            break

# ── Redis helpers ─────────────────────────────────────────────────────────────
def _is_paused(r: redis.Redis) -> bool:
    return r.get('pipeline:state') == 'paused'

def _mark_done(r: redis.Redis, job: dict, mp3_path: Path):
    book_id   = job['book_id']
    chunk_idx = job['chunk_idx']
    r.hset(f'chunk:{book_id}:{chunk_idx}', mapping={
        'status':    'done',
        'mp3_path':  str(mp3_path),
        'worker_id': WORKER_ID,
    })
    done_count = r.hincrby(f'book:{book_id}', 'done_chunks', 1)
    total      = int(r.hget(f'book:{book_id}', 'total_chunks') or 0)
    if total and done_count >= total:
        title    = r.hget(f'book:{book_id}', 'title') or book_id
        out_dir  = str(OUTPUT_DIR_W / book_id)
        r.hset(f'book:{book_id}', 'status', 'merging')
        r.lpush('pipeline:done', json.dumps({
            'book_id': book_id,
            'title':   title,
            'total':   total,
            'out_dir': out_dir,
        }))
        log.info('Book %s complete — queued for merge', book_id)

def _mark_error(r: redis.Redis, job: dict, err: str):
    book_id   = job['book_id']
    chunk_idx = job['chunk_idx']
    r.hset(f'chunk:{book_id}:{chunk_idx}', mapping={
        'status': 'error',
        'error':  err[:500],
    })

# ── main loop ─────────────────────────────────────────────────────────────────
log.info('Fish Speech worker %s started. Listening on pipeline:tts...', WORKER_ID)

_running = True
def _handle_sigint(*_):
    global _running
    log.info('Interrupted — finishing current chunk then stopping.')
    _running = False
signal.signal(signal.SIGINT, _handle_sigint)

while _running:
    try:
        r = redis.from_url(REDIS_URL, decode_responses=True, socket_connect_timeout=5)

        _heartbeat.labels(worker_id=WORKER_ID).set(time.time())
        _set_status('Idle')

        while _is_paused(r):
            log.info('Pipeline paused — waiting...')
            time.sleep(10)

        item = r.brpop('pipeline:tts', timeout=5)
        if item is None:
            continue

        _, raw = item
        job = json.loads(raw)
        book_id   = job['book_id']
        chunk_idx = job['chunk_idx']
        text      = job['text']
        speed     = float(job.get('speed', 1.0))

        log.info('[%s] chunk %d  (%d chars)', book_id, chunk_idx, len(text))
        _set_status('Processing')
        r.hset(f'chunk:{book_id}:{chunk_idx}', mapping={
            'status': 'processing', 'worker_id': WORKER_ID,
        })

        t0 = time.time()
        try:
            mp3_bytes = synthesize_text(text, speed=speed)
        except Exception as e:
            elapsed = time.time() - t0
            log.error('[%s] chunk %d synthesis failed (%.1fs): %s', book_id, chunk_idx, elapsed, e)
            _mark_error(r, job, str(e))
            _jobs_total.labels(status='error', worker_id=WORKER_ID).inc()
            _set_status('Error')
            continue

        elapsed = time.time() - t0
        _api_latency.labels(worker_id=WORKER_ID).observe(elapsed)

        mp3_path = _output_path(book_id, chunk_idx)
        mp3_path.write_bytes(mp3_bytes)
        _flush_spool(book_id)

        _mark_done(r, job, mp3_path)
        _jobs_total.labels(status='success', worker_id=WORKER_ID).inc()
        log.info('[%s] chunk %d done in %.1fs → %s', book_id, chunk_idx, elapsed, mp3_path.name)

    except redis.exceptions.ConnectionError as e:
        log.warning('Redis connection error: %s — retrying in %ds', e, SMB_RETRY)
        _set_status('Error')
        time.sleep(SMB_RETRY)
    except KeyboardInterrupt:
        break
    except Exception as e:
        log.exception('Unexpected error: %s', e)
        _set_status('Error')
        time.sleep(5)

log.info('Worker stopped.')